In [40]:
import pandas as pd
import os
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.max_columns", None)

In [41]:
DATA_PATH = "../data/processed/station_hour_sorted.csv"
df = pd.read_csv(DATA_PATH)
df[["StationId", "Datetime"]].head(20)

C:\Users\Yosua Triantara\AppData\Local\Temp\ipykernel_4536\319879956.py:2: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATA_PATH)


,StationId,Datetime
0,AP001,2017-11-24 17:00:00
1,AP001,2017-11-24 18:00:00
2,AP001,2017-11-24 19:00:00
3,AP001,2017-11-24 20:00:00
4,AP001,2017-11-24 21:00:00
5,AP001,2017-11-24 22:00:00
6,AP001,2017-11-24 23:00:00
7,AP001,2017-11-25 00:00:00
8,AP001,2017-11-25 01:00:00
9,AP001,2017-11-25 02:00:00


## Cek Missing Values setiap Kolom

In [42]:
features = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO",
    "NO2",
    "NOx",
    "NH3",
    "CO",
    "SO2",
    "O3",
    "Benzene",
    "Toluene",
    "Xylene"
]

In [43]:
features = [f for f in features if f in df.columns]
missing_summary = pd.DataFrame({
    "Missing Count": df[features].isna().sum(),
    "Missing (%)": (df[features].isna().mean()*100).round(2)
})

missing_summary = missing_summary.sort_values(
    "Missing (%)",
    ascending=False
)

missing_summary

,Missing Count,Missing (%)
Xylene,2075104,80.15
NH3,1236618,47.76
PM10,1119252,43.23
Toluene,1042366,40.26
Benzene,861579,33.28
SO2,742737,28.69
O3,725973,28.04
PM2.5,647689,25.02
AQI,570190,22.02
NO,553711,21.39


### Buat Report

In [44]:
report_dir = "../reports"
os.makedirs(report_dir, exist_ok=True)
report_path = f"{report_dir}/missing_value_report.md"

with open(report_path, "w", encoding="utf-8") as f:

    f.write("# Missing Value Analysis Report\n\n")

    f.write("## Ringkasan\n\n")
    f.write(f"- Jumlah Data : {len(df):,}\n")
    f.write(f"- Jumlah Fitur Dianalisis : {len(features)}\n\n")

    f.write("## Missing Value\n\n")

    f.write("| Feature | Missing | Percentage |\n")
    f.write("|---------|--------:|----------- |\n")

    for idx, row in missing_summary.iterrows():
        f.write(
            f"| {idx} | {row['Missing Count']} | "
            f"{row['Missing (%)']:.2f}% |\n"
        )
        
print("Report berhasil disimpan.")

Report berhasil disimpan.


## Interpolasi

melakukan pengecekan kembali jumlah missing value sebelum di interpolasi

In [45]:
missing_report = pd.DataFrame({
    "Before": df.isnull().sum()
})
missing_report = missing_report[
    missing_report["Before"] > 0
]
missing_report

,Before
PM2.5,647689
PM10,1119252
NO,553711
NO2,528973
NOx,490808
NH3,1236618
CO,499302
SO2,742737
O3,725973
Benzene,861579


Fitur dengan missing value lebih dari 45% tidak digunakan. Sedangkan Benzene dan Toluene tidak digunakan karena berkorelasi rendah terhadap fitur target. Namun, fitur PM10 tetap digunakan karena berkorelasi tinggi terhadap fitur target

In [46]:
candidate_features = [
    "PM2.5", "PM10", "NO", "NO2", "NOx",
    "NH3", "CO", "SO2", "O3",
    "Benzene", "Toluene", "Xylene"
]

target_col = "AQI"

metadata_cols = ["StationId", "StationName", "City", "State"]

### Pengecekan Gap Missing Value

Sebagai dasar penentuan limit pada tahap interpolasi linear, dilakukan pengecekan gap missing value berdasarkan masing masing stasiun

In [47]:
def consecutive_missing_lengths(series):
    """
    Mengembalikan daftar panjang missing value berturut-turut.
    """
    is_missing = series.isna()

    groups = (
        is_missing != is_missing.shift()
    ).cumsum()

    lengths = (
        is_missing
        .groupby(groups)
        .sum()
    )

    return lengths[lengths > 0].tolist()

In [48]:
gap_stats = []

for feature in candidate_features:

    gaps = []

    for _, group in df.groupby("StationId"):
        gaps.extend(consecutive_missing_lengths(group[feature]))

    gaps = pd.Series(gaps)

    gap_stats.append({
        "Feature": feature,
        "Median": gaps.median(),
        "90%": gaps.quantile(0.90),
        "95%": gaps.quantile(0.95),
        "99%": gaps.quantile(0.99),
        "Max": gaps.max()
    })

gap_stats = pd.DataFrame(gap_stats)

gap_stats

,Feature,Median,90%,95%,99%,Max
0,PM2.5,2.0,18.0,35.0,164.72,40736
1,PM10,2.0,18.0,35.0,168.00,48192
2,NO,2.0,20.0,41.0,172.18,26678
3,NO2,2.0,22.0,45.0,200.12,27683
4,NOx,2.0,21.0,40.0,167.33,37236
5,NH3,2.0,21.0,41.0,192.55,48192
6,CO,2.0,19.0,35.0,128.78,37236
7,SO2,2.0,15.0,26.0,143.00,48192
8,O3,2.0,20.0,40.0,193.00,48192
9,Benzene,3.0,23.0,47.0,259.80,48192


Analisis consecutive missing values menunjukkan bahwa,
- sebagian besar missing value muncul dalam bentuk gap yang relatif pendek. 
- Nilai median panjang gap pada seluruh fitur adalah 2 observasi, sedangkan 95% gap memiliki panjang tidak lebih dari 45 observasi.  
- beberapa gap yang sangat panjang dengan panjang maksimum mencapai lebih dari 40.000 observasi sehingga tidak dilakukan interpolasi linear karena berpotensi menghasilkan estimasi yang tidak representatif

In [49]:
recommended_limit = int(
    gap_stats["95%"].max()
)

print(f"Recommended interpolation limit: {recommended_limit} hours")

Recommended interpolation limit: 76 hours


- missing value candidate sebelum interpolasi

In [50]:
missing_before = df[candidate_features].isna().sum()
missing_before

PM2.5       647689
PM10       1119252
NO          553711
NO2         528973
NOx         490808
NH3        1236618
CO          499302
SO2         742737
O3          725973
Benzene     861579
Toluene    1042366
Xylene     2075104
dtype: int64

- interpolasi candidate feature

In [51]:
df.head()

,StationId,Datetime,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
0,AP001,2017-11-24 17:00:00,60.50,98.00,2.35,30.80,18.25,8.50,0.1,11.85,126.40,0.1,6.10,0.10,NaN,NaN
1,AP001,2017-11-24 18:00:00,65.50,111.25,2.70,24.20,15.07,9.77,0.1,13.17,117.12,0.1,6.25,0.15,NaN,NaN
2,AP001,2017-11-24 19:00:00,80.00,132.00,2.10,25.18,15.15,12.02,0.1,12.08,98.98,0.2,5.98,0.18,NaN,NaN
3,AP001,2017-11-24 20:00:00,81.50,133.25,1.95,16.25,10.23,11.58,0.1,10.47,112.20,0.2,6.72,0.10,NaN,NaN
4,AP001,2017-11-24 21:00:00,75.25,116.00,1.43,17.48,10.43,12.03,0.1,9.12,106.35,0.2,5.75,0.08,NaN,NaN


In [52]:
df["Datetime"] = pd.to_datetime(df["Datetime"])

In [53]:
df = (
    df.sort_values(["StationId", "Datetime"])
      .set_index("Datetime")
)

In [54]:
df.head()

,StationId,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket
Datetime,,,,,,,,,,,,,,,
2017-11-24 17:00:00,AP001,60.50,98.00,2.35,30.80,18.25,8.50,0.1,11.85,126.40,0.1,6.10,0.10,NaN,NaN
2017-11-24 18:00:00,AP001,65.50,111.25,2.70,24.20,15.07,9.77,0.1,13.17,117.12,0.1,6.25,0.15,NaN,NaN
2017-11-24 19:00:00,AP001,80.00,132.00,2.10,25.18,15.15,12.02,0.1,12.08,98.98,0.2,5.98,0.18,NaN,NaN
2017-11-24 20:00:00,AP001,81.50,133.25,1.95,16.25,10.23,11.58,0.1,10.47,112.20,0.2,6.72,0.10,NaN,NaN
2017-11-24 21:00:00,AP001,75.25,116.00,1.43,17.48,10.43,12.03,0.1,9.12,106.35,0.2,5.75,0.08,NaN,NaN


In [55]:
# Tahap 1
# Forward / Backward Fill
# hanya untuk gap sangat pendek (<=2 jam)

for col in candidate_features:
    df[col] = (
        df.groupby("StationId")[col]
          .transform(lambda s: s.ffill(limit=2).bfill(limit=2))
    )

# Tahap 2
# Time Interpolation
# hanya untuk gap sedang (<=24 jam)

for col in candidate_features:
    df[col] = (
        df.groupby("StationId")[col]
          .apply(
              lambda s: s.interpolate(
                  method="time",
                  limit=24,
                  limit_area="inside"
              )
          )
          .reset_index(level=0, drop=True)
    )

# Tahap 3
# Median tiap station
for col in candidate_features:
    median_station = (
        df.groupby("StationId")[col]
          .transform("median")
    )

    df[col] = df[col].fillna(median_station)

# Kembalikan Datetime menjadi kolom biasa
df = df.reset_index()

# Hapus baris yang masih memiliki missing
df = df.dropna(subset=candidate_features)
df = df.dropna(subset=target_col)

print("="*50)
print("Missing setelah imputasi")
print("="*50)
print(df[candidate_features + [target_col]].isnull().sum())

print(f"\nTotal data : {len(df):,}")

Missing setelah imputasi
PM2.5      0
PM10       0
NO         0
NO2        0
NOx        0
NH3        0
CO         0
SO2        0
O3         0
Benzene    0
Toluene    0
Xylene     0
AQI        0
dtype: int64

Total data : 304,644


In [56]:
missing_after = df[candidate_features].isna().sum()
comparison = pd.DataFrame({
    "Before": missing_before,
    "After": missing_after
})
comparison["Recovered"] = (
    comparison["Before"] -
    comparison["After"]
)
comparison["Recovery (%)"] = (
    comparison["Recovered"] /
    comparison["Before"] * 100
).round(2)

comparison

,Before,After,Recovered,Recovery (%)
PM2.5,647689,0,647689,100.0
PM10,1119252,0,1119252,100.0
NO,553711,0,553711,100.0
NO2,528973,0,528973,100.0
NOx,490808,0,490808,100.0
NH3,1236618,0,1236618,100.0
CO,499302,0,499302,100.0
SO2,742737,0,742737,100.0
O3,725973,0,725973,100.0
Benzene,861579,0,861579,100.0


Setelah dilakukan interpolasi linear dengan batas maksimum 45 observasi, masih terdapat sejumlah missing value yang berasal dari gap panjang atau berada pada awal dan akhir deret waktu sehingga tidak memenuhi kriteria untuk diinterpolasi

### cek missing value setelah interpolasi

In [57]:
missing_after = df[candidate_features + ["AQI"]].isnull().sum()

missing_after = pd.DataFrame({
    "Missing Count": missing_after,
    "Missing (%)": (
        missing_after / len(df) * 100
    ).round(2)
})

missing_after

,Missing Count,Missing (%)
PM2.5,0,0.0
PM10,0,0.0
NO,0,0.0
NO2,0,0.0
NOx,0,0.0
NH3,0,0.0
CO,0,0.0
SO2,0,0.0
O3,0,0.0
Benzene,0,0.0


In [58]:
rows_with_missing = df[
    candidate_features + ["AQI"]
].isnull().any(axis=1).sum()

print(f"Jumlah baris yang masih memiliki missing value : {rows_with_missing:,}")
print(f"Persentase : {rows_with_missing/len(df)*100:.2f}%")

Jumlah baris yang masih memiliki missing value : 0
Persentase : 0.00%


In [59]:
df[candidate_features].isna().mean().sort_values(ascending=False)

PM2.5      0.0
PM10       0.0
NO         0.0
NO2        0.0
NOx        0.0
NH3        0.0
CO         0.0
SO2        0.0
O3         0.0
Benzene    0.0
Toluene    0.0
Xylene     0.0
dtype: float64

In [60]:
missing_pattern = (
    df[candidate_features]
      .isna()
      .sum(axis=1)
      .value_counts()
      .sort_index()
)

print(missing_pattern)

0    304644
Name: count, dtype: int64


In [61]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 304644 entries, 16 to 2547206
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   Datetime    304644 non-null  datetime64[ns]
 1   StationId   304644 non-null  object        
 2   PM2.5       304644 non-null  float64       
 3   PM10        304644 non-null  float64       
 4   NO          304644 non-null  float64       
 5   NO2         304644 non-null  float64       
 6   NOx         304644 non-null  float64       
 7   NH3         304644 non-null  float64       
 8   CO          304644 non-null  float64       
 9   SO2         304644 non-null  float64       
 10  O3          304644 non-null  float64       
 11  Benzene     304644 non-null  float64       
 12  Toluene     304644 non-null  float64       
 13  Xylene      304644 non-null  float64       
 14  AQI         304644 non-null  float64       
 15  AQI_Bucket  304644 non-null  object        
dtypes: da

### Buat final dataset

In [63]:
df_clean = df.copy()

In [64]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "station_hour_clean.csv"

df_clean.to_csv(output_path, index=False)

print(f"Dataset berhasil disimpan di: {output_path}")

Dataset berhasil disimpan di: ..\data\processed\station_hour_clean.csv
